In [10]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Activation, Dropout, MaxPooling1D, Flatten, Dense

# Recreate the EXACT same model architecture
def create_emotion_model():
    model = Sequential()

    model.add(Conv1D(128, 5, padding='same',
                     input_shape=(40, 1)))
    model.add(Activation('relu'))
    model.add(Dropout(0.1))
    model.add(MaxPooling1D(pool_size=8))
    model.add(Conv1D(128, 5, padding='same'))
    model.add(Activation('relu'))
    model.add(Dropout(0.1))
    model.add(Flatten())
    model.add(Dense(8))
    model.add(Activation('softmax'))

    return model

# Create model
model = create_emotion_model()

# Load weights only (not the whole model)
model.load_weights('/content/Emotion_Voice_Detection_Model.h5')

print("✅ Model recreated and weights loaded!")

✅ Model recreated and weights loaded!


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
# If you saved your test data
import joblib
X_test = joblib.load('/content/X.joblib')
y_test = joblib.load('/content/y.joblib')

# Reshape for CNN
X_test_reshaped = np.expand_dims(X_test, axis=2)

In [12]:
# Get prediction probabilities
predictions = model.predict(X_test_reshaped)

# Get the predicted emotion class
predicted_class = np.argmax(predictions, axis=1)[0]

# Get confidence (probability)
confidence = np.max(predictions) * 100

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 383ms/step


In [13]:
# Emotion mapping
emotion_dict = {
    0: "neutral",
    1: "calm",
    2: "happy",
    3: "sad",
    4: "angry",
    5: "fearful",
    6: "disgust",
    7: "surprised"
}

# Get emotion name
predicted_emotion = emotion_dict[predicted_class]

print(f"Predicted Emotion: {predicted_emotion}")
print(f"Confidence: {confidence:.2f}%")
print(f"Raw probabilities: {predictions[0]}")

Predicted Emotion: neutral
Confidence: 100.00%
Raw probabilities: [9.9932677e-01 5.3358806e-04 6.0617281e-06 5.7675861e-05 1.4198166e-09
 7.7253171e-06 3.1313487e-08 6.8158355e-05]


In [20]:
# ========== INSTALL REQUIRED PACKAGES ==========
!pip install resampy librosa numpy tensorflow soundfile -q

# ========== IMPORT EVERYTHING ==========
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from tensorflow import keras
import warnings
warnings.filterwarnings('ignore')
import os
print("✅ All packages imported successfully!")

# ========== LOAD MODEL ==========
def load_emotion_model(model_path='/content/Emotion_Voice_Detection_Model.h5'):
    """Load the emotion detection model"""
    try:
        # Try loading directly
        model = keras.models.load_model(model_path)
        print("✅ Model loaded directly!")
    except Exception as e:
        # If that fails, recreate architecture
        print(f"⚠️ Loading failed: {e}")
        print("Recreating model architecture...")
        model = keras.Sequential([
            keras.layers.Input(shape=(40, 1)),
            keras.layers.Conv1D(128, 5, padding='same'),
            keras.layers.Activation('relu'),
            keras.layers.Dropout(0.1),
            keras.layers.MaxPooling1D(pool_size=8),
            keras.layers.Conv1D(128, 5, padding='same'),
            keras.layers.Activation('relu'),
            keras.layers.Dropout(0.1),
            keras.layers.Flatten(),
            keras.layers.Dense(8),
            keras.layers.Activation('softmax')
        ])
        model.load_weights(model_path)
        model.compile(loss='sparse_categorical_crossentropy',
                      optimizer='adam',
                      metrics=['accuracy'])
        print("✅ Model recreated and weights loaded!")

    # Show model info
    print(f"\n📊 Model Info:")
    print(f"Input shape: {model.input_shape}")
    print(f"Output shape: {model.output_shape}")

    return model

# Load the model
model = load_emotion_model()

# ========== TEST FUNCTION ==========
def test_emotion_model(audio_path):
    """
    Test the emotion detection model on a single audio file
    """
    print("\n" + "="*60)
    print(f"🎵 Testing: {audio_path}")
    print("="*60)

    try:
        # Check if file exists
        if not os.path.exists(audio_path):
            print(f"❌ File not found: {audio_path}")
            return None, None

        # Step 1: Load audio (simpler method without res_type)
        print("📥 Loading audio file...")
        X, sample_rate = librosa.load(audio_path, sr=None)  # sr=None keeps original rate

        print(f"   Audio length: {len(X)} samples")
        print(f"   Sample rate: {sample_rate} Hz")
        print(f"   Duration: {len(X)/sample_rate:.2f} seconds")

        # Step 2: Extract MFCC features
        print("🔍 Extracting MFCC features...")
        mfccs = librosa.feature.mfcc(y=X, sr=sample_rate, n_mfcc=40)
        mfccs_mean = np.mean(mfccs.T, axis=0)

        print(f"   MFCC shape: {mfccs.shape}")
        print(f"   MFCC mean shape: {mfccs_mean.shape}")

        # Step 3: Reshape for model
        mfccs_reshaped = mfccs_mean.reshape(1, 40, 1)

        # Step 4: Predict
        print("🤖 Making prediction...")
        predictions = model.predict(mfccs_reshaped, verbose=0)
        predicted_class = np.argmax(predictions, axis=1)[0]
        confidence = np.max(predictions) * 100

        # Step 5: Emotion mapping
        emotion_dict = {
            0: "NEUTRAL",
            1: "CALM",
            2: "HAPPY",
            3: "SAD",
            4: "ANGRY",
            5: "FEARFUL",
            6: "DISGUST",
            7: "SURPRISED"
        }

        # Display results
        print("\n📊 EMOTION PROBABILITIES:")
        print("-"*50)
        emotions = ["Neutral", "Calm", "Happy", "Sad",
                   "Angry", "Fearful", "Disgust", "Surprised"]

        for i, emotion_name in enumerate(emotions):
            prob_percent = predictions[0][i] * 100
            star = " ⭐" if i == predicted_class else ""
            bar = "█" * int(prob_percent / 5)  # Visual bar
            print(f"  {emotion_name:10}: {prob_percent:6.2f}% {bar}{star}")

        print("-"*50)
        print(f"🎯 FINAL PREDICTION: {emotion_dict[predicted_class]}")
        print(f"📈 CONFIDENCE: {confidence:.2f}%")
        print("="*60)

        return emotion_dict[predicted_class], confidence

    except Exception as e:
        print(f"❌ ERROR: {str(e)}")
        import traceback
        traceback.print_exc()
        return None, None

# ========== TEST WITH SAMPLE AUDIO ==========
# First, let's create or find a test audio

print("\n" + "="*60)
print("🎤 CREATING TEST AUDIO")
print("="*60)

test_emotion_model('/content/narendra-modi-ji-voice-made-with-Voicemod (1).mp3')

✅ All packages imported successfully!
⚠️ Loading failed: Kernel shape must have the same length as input, but received kernel of shape (5, 1, 128) and input of shape (None, None, 40, 1).
Recreating model architecture...
✅ Model recreated and weights loaded!

📊 Model Info:
Input shape: (None, 40, 1)
Output shape: (None, 8)

🎤 CREATING TEST AUDIO

🎵 Testing: /content/narendra-modi-ji-voice-made-with-Voicemod (1).mp3
📥 Loading audio file...
   Audio length: 1369728 samples
   Sample rate: 48000 Hz
   Duration: 28.54 seconds
🔍 Extracting MFCC features...
   MFCC shape: (40, 2676)
   MFCC mean shape: (40,)
🤖 Making prediction...

📊 EMOTION PROBABILITIES:
--------------------------------------------------
  Neutral   :   0.00% 
  Calm      :   0.00% 
  Happy     :   0.00% 
  Sad       :   0.00% 
  Angry     :   0.00% 
  Fearful   : 100.00% ████████████████████ ⭐
  Disgust   :   0.00% 
  Surprised :   0.00% 
--------------------------------------------------
🎯 FINAL PREDICTION: FEARFUL
📈 CONF

('FEARFUL', np.float32(100.0))